In [ ]:
# ==========================================================
# TELECOM X -
# Challenge Data Science - Alura LATAM
# ==========================================================

"""
Proyecto: Telecom X - Preparación de datos mediante ETL

Autor: Nico
Objetivo:
Aplicar el proceso ETL (Extract, Transform, Load) para preparar
los datos de clientes de Telecom X para análisis posteriores.

Proceso:

Extract
Obtención de datos desde una fuente externa en formato JSON.

Transform
Limpieza, normalización y transformación de datos.

Load
Generación de un dataset limpio listo para análisis y visualización.
"""

# ==========================================================
# 1 IMPORTAR LIBRERÍAS
# ==========================================================

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

# ==========================================================
# 2 EXTRACT - EXTRACCIÓN DE DATOS
# ==========================================================

print("Iniciando proceso de extracción de datos")

url = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"

response = requests.get(url)
data = response.json()

df = pd.json_normalize(data)

print("Datos cargados correctamente")

# ==========================================================
# 3 EXPLORACIÓN INICIAL
# ==========================================================

print("\nDimensiones del dataset")
print(df.shape)

print("\nColumnas del dataset")
print(df.columns)

print("\nPrimeros registros")
display(df.head())

# ==========================================================
# 4 NORMALIZACIÓN DEL JSON
# ==========================================================

print("\nNormalizando estructuras JSON")

df_customer = pd.json_normalize(df['customer'])
df_phone = pd.json_normalize(df['phone'])
df_internet = pd.json_normalize(df['internet'])
df_account = pd.json_normalize(df['account'])

print("Normalización completada")

# ==========================================================
# 5 UNIFICACIÓN DEL DATASET
# ==========================================================

df_final = pd.concat([
    df[['customerID', 'Churn']],
    df_customer,
    df_phone,
    df_internet,
    df_account
], axis=1)

print("\nDataset unificado")

display(df_final.head())

# ==========================================================
# 6 ANÁLISIS DE CALIDAD DE DATOS
# ==========================================================

print("\nRevisión de valores nulos")

missing = df_final.isnull().sum()

print(missing)

# porcentaje de datos faltantes

missing_percent = (missing / len(df_final)) * 100

print("\nPorcentaje de datos faltantes")

print(missing_percent)

# ==========================================================
# 7 LIMPIEZA DE DATOS
# ==========================================================

print("\nIniciando limpieza de datos")

df_final.replace(" ", np.nan, inplace=True)

# eliminar duplicados

df_final.drop_duplicates(inplace=True)

print("Duplicados eliminados")

# ==========================================================
# 8 TRANSFORMACIÓN DE VARIABLES
# ==========================================================

print("\nTransformando variables categóricas")

map_si_no = {
    'Yes':1,
    'No':0,
    'No internet service':0,
    'No phone service':0
}

for col in df_final.columns:
    if df_final[col].dtype == 'object':
        df_final[col] = df_final[col].replace(map_si_no)

print("Transformación completada")

# ==========================================================
# 9 CONVERSIÓN DE TIPOS
# ==========================================================

print("\nConvirtiendo columnas numéricas")

df_final['Charges.Monthly'] = pd.to_numeric(df_final['Charges.Monthly'], errors='coerce')

df_final['Charges.Total'] = pd.to_numeric(df_final['Charges.Total'], errors='coerce')

# ==========================================================
# 10 RENOMBRAR COLUMNAS
# ==========================================================

print("\nRenombrando columnas")

df_final.rename(columns={
    'customerID':'ClienteID',
    'Churn':'Abandono',
    'gender':'Genero',
    'tenure':'MesesContrato',
    'Charges.Monthly':'CobroMensual',
    'Charges.Total':'CobroTotal'
}, inplace=True)

print("Columnas actualizadas")

print("\nColumnas finales")

print(df_final.columns)

# ==========================================================
# 11 ESTADÍSTICAS GENERALES
# ==========================================================

print("\nEstadísticas descriptivas")

display(df_final.describe())

# ==========================================================
# 12 ANÁLISIS DE ABANDONO
# ==========================================================

print("\nDistribución de abandono")

print(df_final['Abandono'].value_counts())

# porcentaje

churn_percent = df_final['Abandono'].value_counts(normalize=True) * 100

print("\nPorcentaje de abandono")

print(churn_percent)

# ==========================================================
# 13 VISUALIZACIONES
# ==========================================================

# gráfico abandono

plt.figure()

df_final['Abandono'].value_counts().plot(kind='bar')

plt.title("Distribución de abandono de clientes")

plt.xlabel("Abandono")

plt.ylabel("Cantidad de clientes")

plt.show()

# gráfico distribución mensual

plt.figure()

df_final['CobroMensual'].hist(bins=30)

plt.title("Distribución de cobros mensuales")

plt.xlabel("Cobro mensual")

plt.ylabel("Frecuencia")

plt.show()

# relación meses contrato

plt.figure()

plt.scatter(df_final['MesesContrato'], df_final['CobroMensual'])

plt.title("Relación entre meses de contrato y cobro mensual")

plt.xlabel("Meses de contrato")

plt.ylabel("Cobro mensual")

plt.show()

# ==========================================================
# 14 LOAD - EXPORTACIÓN DE DATOS
# ==========================================================

print("\nGuardando dataset limpio")

df_final.to_csv("telecomx_clean.csv", index=False)

print("Archivo telecomx_clean.csv creado")

# ==========================================================
# 15 VALIDACIÓN FINAL
# ==========================================================

print("\nVista final del dataset")

display(df_final.head())

print("\nProceso ETL completado exitosamente")